# Práctica 5 — Análisis funcional: knockout de MYCN en neuroblastoma

**Módulo de Transcriptómica — Máster en Bioinformática**

En esta práctica partimos de los resultados de un análisis de expresión diferencial ya realizado (RNA-seq de células de neuroblastoma con knockout de **MYCN** mediante CRISPR) y realizamos el **análisis funcional** completo:

1. **ORA** (Over-Representation Analysis) con Enrichr sobre los genes significativos.
2. **GSEA pre-ranked** a partir de una lista de genes ordenada.
3. **GSEA clásico** a partir de la matriz de cuentas normalizadas.
4. **ssGSEA** para obtener una puntuación por ruta y por muestra.

**Diseño experimental:**

| Grupo | Muestras |
|---|---|
| Control (sgRNA contra AAVS1) | `sgAAVS1-1` … `sgAAVS1-5` |
| Knockout MYCN | `sgMYCN-1` … `sgMYCN-5` |

**Datos de partida:**

- `degs.csv`: resultados del análisis diferencial (16.235 genes, columnas `baseMean`, `log2FoldChange`, `pvalue`, `padj`…). El contraste es **KO vs Control** (log2FC > 0 = sube en KO).
- `normalized_counts.csv`: matriz de cuentas normalizadas (genes × 10 muestras).

> **Nota sobre los datos**: el gen *MYCN* no aparece en estas matrices (fue filtrado en pasos previos del pipeline), por lo que no podremos usar su propia expresión como control. Sí podremos evaluar el efecto del knockout a través de sus **genes diana** (p. ej. los *Hallmark MYC targets*), que es biológicamente más informativo.

## 1. Instalación y carga de datos

Instalamos `gseapy`, la librería que usaremos para ORA, GSEA y ssGSEA. Fijamos `pandas==2.2.3` por compatibilidad con Colab.

In [ ]:
# Instalación de dependencias (ejecutar una sola vez al abrir el notebook)
!pip install -q gseapy "pandas==2.2.3"
print("Instalación completada.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp

pd.set_option("display.max_columns", 20)
print("gseapy", gp.__version__)
print("pandas", pd.__version__)

### Descarga de los datos

Los datos están en el repositorio del curso. Si la descarga falla, sube manualmente `degs.csv` y `normalized_counts.csv` con el panel de archivos de Colab (o con `files.upload()`).

In [ ]:
import urllib.request
from pathlib import Path

DATA_DIR = Path("datos_mycn")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/victorsanchezarevalo/transcriptomica-master/main/practicas/05-analisis-funcional-mycn/datos"

FICHEROS = ["degs.csv", "normalized_counts.csv"]

for f in FICHEROS:
    destino = DATA_DIR / f
    if not destino.exists():
        print(f"Descargando {f} ...")
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", destino)
    print(f"{f}: {destino.stat().st_size / 1e6:.2f} MB")

# Comprobación
for f in FICHEROS:
    assert (DATA_DIR / f).exists(), f"Falta {f}: súbelo manualmente a la carpeta {DATA_DIR}"
print("\nDatos listos.")

## 2. Carga y limpieza de los datos

Los identificadores de gen son **símbolos**, que es lo que esperan las librerías de Enrichr/MSigDB. Puntos importantes de limpieza:

1. Pasamos todos los símbolos a **mayúsculas** y eliminamos posibles versiones tipo Ensembl (por seguridad).
2. En `normalized_counts.csv` hay filas **sin `gene_id`** que hay que eliminar.
3. Hay **símbolos duplicados** (sobre todo RNAs repetitivos como `Y_RNA`): GSEA y ssGSEA requieren IDs únicos, así que los agregamos (suma de cuentas) y en el ranking nos quedamos con el registro más extremo.

In [ ]:
def clean_gene_name(name):
    """Limpia versiones de Ensembl y pasa a mayúsculas."""
    return str(name).split('.')[0].upper().strip()

# ---------- degs.csv (resultados del análisis diferencial) ----------
degs = pd.read_csv(DATA_DIR / "degs.csv")
print(f"degs.csv original: {degs.shape[0]} genes x {degs.shape[1]} columnas")
print(degs.head(3))

degs["gene"] = degs["gene_id"].apply(clean_gene_name)
degs = degs.dropna(subset=["gene_id"])
degs = degs[degs["gene"] != "NAN"]
print(f"\nTras limpieza: {degs.shape[0]} genes")
print(f"Genes con padj NA (filtrados por DESeq2): {degs['padj'].isna().sum()}")

In [ ]:
# ---------- normalized_counts.csv (matriz de expresión) ----------
counts = pd.read_csv(DATA_DIR / "normalized_counts.csv")
print(f"normalized_counts.csv original: {counts.shape[0]} genes x {counts.shape[1]} columnas")
print("Columnas:", list(counts.columns))

# Muestras y grupos
MUESTRAS = list(counts.columns[1:])
CONTROL = [s for s in MUESTRAS if s.startswith("sgAAVS1")]
KO = [s for s in MUESTRAS if s.startswith("sgMYCN")]
print(f"\nControles ({len(CONTROL)}): {CONTROL}")
print(f"KO ({len(KO)}): {KO}")
assert len(CONTROL) == 5 and len(KO) == 5, "El diseño debería ser 5 vs 5"

# Limpieza: eliminar filas sin gene_id y agregar duplicados (suma)
n_na = counts["gene_id"].isna().sum()
counts = counts.dropna(subset=["gene_id"]).copy()
counts["gene"] = counts["gene_id"].apply(clean_gene_name)
counts = counts[counts["gene"] != "NAN"]
print(f"\nFilas sin gene_id eliminadas: {n_na}")

n_dup = counts["gene"].duplicated().sum()
expr = counts.drop(columns=["gene_id"]).groupby("gene")[MUESTRAS].sum()
print(f"Símbolos duplicados agregados (suma): {n_dup}")
print(f"Matriz final de expresión: {expr.shape[0]} genes x {expr.shape[1]} muestras")

### Comprobación rápida: coherencia de los datos

Antes del análisis funcional verificamos que los dos ficheros son coherentes entre sí:

1. El log2FC calculado a mano desde las cuentas normalizadas debe correlacionar con el `log2FoldChange` de `degs.csv`.
2. Un PCA rápido sobre los genes más variables debe separar los dos grupos.

In [ ]:
# 1) Coherencia degs vs normalized_counts
comunes = expr.index.intersection(degs["gene"])
print(f"Genes comunes: {len(comunes)} de {degs.shape[0]} en degs.csv")

log2fc_manual = np.log2((expr.loc[comunes, KO].mean(axis=1) + 1) /
                        (expr.loc[comunes, CONTROL].mean(axis=1) + 1))
degs_idx = degs.drop_duplicates("gene").set_index("gene")
r = np.corrcoef(log2fc_manual, degs_idx.loc[comunes, "log2FoldChange"])[0, 1]
print(f"Correlación log2FC (degs.csv) vs log2FC aproximado (normalized_counts): r = {r:.4f}")
print("→ Valores > 0.95 confirman que ambos ficheros proceden del mismo experimento.")

In [ ]:
# 2) PCA exploratorio con los 5000 genes más variables
from sklearn.decomposition import PCA

log_expr = np.log2(expr + 1)
top_var = log_expr.var(axis=1).sort_values(ascending=False).head(5000).index
X = log_expr.loc[top_var].T  # muestras x genes

pca = PCA(n_components=2)
coords = pca.fit_transform(X)
var_exp = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(6, 5))
for grupo, muestras, color in [("Control", CONTROL, "steelblue"), ("KO (sgMYCN)", KO, "firebrick")]:
    idx = [MUESTRAS.index(m) for m in muestras]
    ax.scatter(coords[idx, 0], coords[idx, 1], c=color, label=grupo, s=80, edgecolor="k")
for i, m in enumerate(MUESTRAS):
    ax.annotate(m, (coords[i, 0], coords[i, 1]), fontsize=8, xytext=(5, 4), textcoords="offset points")
ax.set_xlabel(f"PC1 ({var_exp[0]:.1f} % varianza)")
ax.set_ylabel(f"PC2 ({var_exp[1]:.1f} % varianza)")
ax.set_title("PCA exploratorio (log2 cuentas normalizadas, 5000 genes más variables)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. ORA: análisis de sobrerrepresentación con Enrichr

El ORA toma la lista de genes **significativos** (FDR < 0.05 y |log2FC| > 1) y comprueba qué rutas/terminologías están sobrerrepresentadas respecto al azar. Analizamos por separado:

- Genes **UP en KO** (log2FC > 1)
- Genes **DOWN en KO** (log2FC < −1)

Usamos tres colecciones de Enrichr/MSigDB:

- `GO_Biological_Process_2023`
- `KEGG_2021_Human`
- `MSigDB_Hallmark_2020`

> **Pregunta guía**: ¿qué procesos esperarías ver alterados al eliminar MYCN, un factor de transcripción que promueve proliferación y crecimiento celular?

In [ ]:
# Genes diferencialmente expresados (sobre el ranking de genes únicos)
degs_u = degs.copy()
degs_u["padj_f"] = degs_u["padj"].fillna(1)  # padj NA = no significativo

up_genes = degs_u.loc[(degs_u["padj_f"] < 0.05) & (degs_u["log2FoldChange"] > 1), "gene"]
up_genes = up_genes.drop_duplicates().tolist()
down_genes = degs_u.loc[(degs_u["padj_f"] < 0.05) & (degs_u["log2FoldChange"] < -1), "gene"]
down_genes = down_genes.drop_duplicates().tolist()

print(f"Genes UP en KO   (FDR<0.05, log2FC>1):  {len(up_genes)}")
print(f"Genes DOWN en KO (FDR<0.05, log2FC<-1): {len(down_genes)}")

In [ ]:
LIBS = ["GO_Biological_Process_2023", "KEGG_2021_Human", "MSigDB_Hallmark_2020"]
ORA_DIR = Path("resultados") / "ORA"
ORA_DIR.mkdir(parents=True, exist_ok=True)

ora_results = {}  # (direccion, libreria) -> DataFrame

for direccion, genes in [("UP", up_genes), ("DOWN", down_genes)]:
    print(f"\n=== ORA genes {direccion} en KO ({len(genes)} genes) ===")
    enr = gp.enrichr(
        gene_list=genes,
        gene_sets=LIBS,
        organism="human",
        outdir=str(ORA_DIR / direccion),
        cutoff=1.0,  # guardamos todo; filtramos después
    )
    res = enr.results.copy()
    res["Direccion"] = direccion
    ora_results[direccion] = res
    sig = res[res["Adjusted P-value"] < 0.05]
    print(f"Términos con FDR < 0.05: {len(sig)}")
    display(sig.sort_values("Adjusted P-value").head(10)[
        ["Gene_set", "Term", "Adjusted P-value", "Odds Ratio", "Genes"]])

In [ ]:
# Dotplot de los términos más significativos por dirección y librería
for direccion in ["UP", "DOWN"]:
    res = ora_results[direccion]
    res_sig = res[res["Adjusted P-value"] < 0.05].copy()
    if res_sig.empty:
        print(f"Sin términos significativos para {direccion}")
        continue
    res_sig["mlog10FDR"] = -np.log10(res_sig["Adjusted P-value"])
    top = (res_sig.sort_values("Adjusted P-value")
                  .groupby("Gene_set", group_keys=False)
                  .head(8)
                  .sort_values("mlog10FDR"))
    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(top))))
    ax.scatter(top["mlog10FDR"], top["Term"],
               s=60, c=top["Gene_set"].astype("category").cat.codes, cmap="Set2", edgecolor="k")
    ax.set_xlabel("-log10(FDR)")
    ax.set_title(f"ORA — genes {direccion} en KO MYCN (top por librería)")
    plt.tight_layout()
    plt.show()

## 4. GSEA pre-ranked

El ORA solo usa los genes que superan un umbral arbitrario. **GSEA pre-ranked** usa **todos los genes**, ordenados por una métrica que resuma magnitud y significación del cambio, y detecta rutas cuyos genes se concentran en los extremos de la lista.

**Métrica de ranking** (la que optimizamos en el script de referencia):

```
Rank = -log10(padj) * sign(log2FC) + 0.01 * log2FC
```

- `-log10(padj) * sign(log2FC)`: ordena por significación con la dirección del cambio.
- `+ 0.01 * log2FC`: desempata genes con el mismo padj (GSEA funciona mal con muchos empates).
- Los `padj` NA se sustituyen por 1 (genes no testeados → Rank ≈ 0) y los `padj = 0` por 1e-300.

> Alternativa habitual: usar la columna `stat` de DESeq2 como ranking. Podéis comparar ambos en las preguntas finales.

In [ ]:
def make_rank_list(df):
    """Crea la serie de ranking optimizada para evitar empates y duplicados."""
    df = df.copy()
    df["padj"] = df["padj"].replace(0, 1e-300).fillna(1)
    df["Rank"] = (-np.log10(df["padj"]) * np.sign(df["log2FoldChange"])) + (df["log2FoldChange"] * 0.01)
    # Si un símbolo aparece varias veces, nos quedamos con el Rank más extremo
    df = df.sort_values("Rank", key=np.abs, ascending=False).drop_duplicates("gene")
    return df.sort_values("Rank", ascending=False).set_index("gene")["Rank"]

rnk = make_rank_list(degs)
print(f"Genes en el ranking: {len(rnk)} (únicos, sin empates: {rnk.duplicated().sum()})")
print("\nTop 5 (arriba = UP en KO):")
print(rnk.head())
print("\nBottom 5 (DOWN en KO):")
print(rnk.tail())

In [ ]:
PRE_DIR = Path("resultados") / "GSEA_prerank"

pre_res = gp.prerank(
    rnk=rnk,
    gene_sets="MSigDB_Hallmark_2020",
    threads=4,
    min_size=5,
    max_size=1000,
    permutation_num=1000,
    outdir=str(PRE_DIR),
    format="png",
    seed=6,
    verbose=True,
)

pre_df = pre_res.res2d.sort_values("FDR q-val")
print("\n=== Resultados GSEA pre-ranked (Hallmark) ===")
display(pre_df[["Term", "ES", "NES", "NOM p-val", "FDR q-val", "FWER p-val"]])

In [ ]:
# Tabla resumen de rutas significativas
sig_pre = pre_df[pre_df["FDR q-val"].astype(float) < 0.05].copy()
print(f"Rutas con FDR < 0.05: {len(sig_pre)} de {len(pre_df)}")

sig_pre["Direccion"] = np.where(sig_pre["NES"].astype(float) > 0, "UP en KO", "DOWN en KO")
display(sig_pre[["Term", "NES", "FDR q-val", "Direccion"]])

# Dotplot NES
plot_df = sig_pre.copy()
plot_df["mlog10FDR"] = -np.log10(plot_df["FDR q-val"].astype(float).clip(lower=1e-10))
plot_df = plot_df.sort_values("NES", key=lambda s: s.astype(float))
fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(plot_df))))
colors = plot_df["NES"].astype(float).map(lambda x: "firebrick" if x > 0 else "steelblue")
ax.barh(plot_df["Term"], plot_df["NES"].astype(float), color=colors, edgecolor="k")
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("NES (Normalized Enrichment Score)")
ax.set_title("GSEA pre-ranked — Hallmark (FDR < 0.05)")
plt.tight_layout()
plt.show()

In [ ]:
# Enrichment plot de las rutas MYC targets (las más relevantes biológicamente)
for term in pre_df["Term"]:
    if "MYC" in term.upper():
        print(term)

# Encontrar el índice del término para el plot
term_myc = [t for t in pre_df["Term"] if "MYC_TARGETS_V1" in t.upper()]
term_myc = term_myc[0] if term_myc else pre_df.iloc[0]["Term"]
print(f"\nRepresentando: {term_myc}")

from gseapy import gseaplot
gseaplot(rank_metric=pre_res.ranking, term=term_myc, **pre_res.results[term_myc])
plt.show()

## 5. GSEA clásico

El GSEA clásico parte de la **matriz de expresión completa** y un vector de clases; el propio algoritmo calcula la métrica de ranking (signal-to-noise) y permuta las etiquetas de las muestras para estimar la significación.

- **Vector de clases**: `Control` × 5, `KO` × 5 (¡en el orden de las columnas!).
- **`permutation_type="phenotype"`** es lo correcto estadísticamente, pero con solo 5 vs 5 muestras hay pocas permutaciones posibles (252 como máximo), así que GSEApy puede fallar; en ese caso usamos `"gene_set"` como fallback (es lo que hace el modo pre-ranked por debajo).

In [ ]:
# gseapy espera: genes como filas, muestras como columnas, con columna 'Name'
gsea_input = expr.copy()
gsea_input.insert(0, "Name", gsea_input.index)

class_vector = ["Control"] * len(CONTROL) + ["KO"] * len(KO)
print("Columnas:", list(gsea_input.columns[1:]))
print("Clases:  ", class_vector)
assert list(gsea_input.columns[1:]) == CONTROL + KO

GSEA_DIR = Path("resultados") / "GSEA_clasico"

try:
    print("\nIntentando permutation_type='phenotype' ...")
    gs_res = gp.gsea(
        data=gsea_input,
        gene_sets="MSigDB_Hallmark_2020",
        cls=class_vector,
        permutation_type="phenotype",
        permutation_num=1000,
        min_size=5,
        max_size=1000,
        seed=6,
        threads=4,
        outdir=str(GSEA_DIR),
        format="png",
        verbose=True,
    )
except Exception as e:
    print(f"Falló 'phenotype' ({type(e).__name__}: {e})")
    print("Usando permutation_type='gene_set' como fallback ...")
    gs_res = gp.gsea(
        data=gsea_input,
        gene_sets="MSigDB_Hallmark_2020",
        cls=class_vector,
        permutation_type="gene_set",
        permutation_num=1000,
        min_size=5,
        max_size=1000,
        seed=6,
        threads=4,
        outdir=str(GSEA_DIR),
        format="png",
        verbose=True,
    )

gsea_df = gs_res.res2d.sort_values("FDR q-val")
print("\n=== Resultados GSEA clásico (Hallmark) ===")
display(gsea_df[["Term", "ES", "NES", "NOM p-val", "FDR q-val"]])

In [ ]:
# Comparación pre-ranked vs clásico: NES por ruta
comp = pre_df[["Term", "NES"]].rename(columns={"NES": "NES_prerank"}).merge(
    gsea_df[["Term", "NES"]].rename(columns={"NES": "NES_clasico"}), on="Term")
comp["NES_prerank"] = comp["NES_prerank"].astype(float)
comp["NES_clasico"] = comp["NES_clasico"].astype(float)

r_nes = comp[["NES_prerank", "NES_clasico"]].corr().iloc[0, 1]
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(comp["NES_prerank"], comp["NES_clasico"], c="teal", edgecolor="k")
ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("NES — GSEA pre-ranked")
ax.set_ylabel("NES — GSEA clásico")
ax.set_title(f"Concordancia de métodos (r = {r_nes:.3f})")
for _, row in comp.iterrows():
    if "MYC" in row["Term"].upper() or "E2F" in row["Term"].upper():
        ax.annotate(row["Term"].replace("HALLMARK_", ""), (row["NES_prerank"], row["NES_clasico"]),
                    fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.tight_layout()
plt.show()

### Heatmap de los *leading-edge genes* de una ruta clave

El **leading edge** es el subconjunto de genes de la ruta que contribuye al enriquecimiento. Lo extraemos de la columna `Lead_genes` y representamos su expresión (z-score por gen) en las 10 muestras. Elegimos la ruta más significativa entre las relacionadas con MYC/ciclo celular, o la primera significativa.

In [ ]:
# Elegir ruta para el heatmap: prioridad a MYC/E2F (biología de MYCN)
gsea_sig = gsea_df[gsea_df["FDR q-val"].astype(float) < 0.25].copy()
prioridad = [t for t in gsea_sig["Term"] if any(k in t.upper() for k in ["MYC", "E2F"])]
term_heat = prioridad[0] if prioridad else gsea_sig.iloc[0]["Term"]
print(f"Ruta seleccionada: {term_heat}")

row = gsea_df[gsea_df["Term"] == term_heat].iloc[0]
lead_genes = [g.strip().upper() for g in str(row["Lead_genes"]).split(";") if g.strip()]
lead_genes = [g for g in lead_genes if g in expr.index]
print(f"Leading-edge genes presentes en la matriz: {len(lead_genes)}")

# Heatmap con z-score por gen
heat = np.log2(expr.loc[lead_genes, CONTROL + KO] + 1)
heat_z = heat.sub(heat.mean(axis=1), axis=0).div(heat.std(axis=1).replace(0, np.nan), axis=0).dropna()

fig, ax = plt.subplots(figsize=(7, max(5, 0.25 * len(heat_z))))
sns.heatmap(heat_z, cmap="RdBu_r", center=0, ax=ax,
            yticklabels=(len(heat_z) <= 60), cbar_kws={"label": "z-score"})
# Separador de grupos
ax.axvline(len(CONTROL), color="k", lw=2)
ax.set_title(f"Leading edge — {term_heat}\n(Control | KO)")
plt.tight_layout()
plt.show()

## 6. ssGSEA: puntuación por ruta y por muestra

A diferencia de GSEA (que compara grupos), **ssGSEA** calcula un *enrichment score* de cada ruta **en cada muestra individual**. El resultado es una matriz **rutas × muestras** que permite:

- Hacer un heatmap de actividad de rutas por muestra.
- Comparar grupos ruta a ruta con un test estadístico (t-test).
- Tratar las rutas como "variables" para PCA u otros análisis.

Usamos la matriz de expresión log2-transformada y las colecciones Hallmark.

In [ ]:
SS_DIR = Path("resultados") / "ssGSEA"

# ssGSEA sobre log2(cuentas+1); los símbolos ya son únicos tras la agregación
ss_res = gp.ssgsea(
    data=log_expr,  # genes x muestras (log2(expr + 1), calculado en la sección 2)
    gene_sets="MSigDB_Hallmark_2020",
    sample_norm_method="rank",
    min_size=5,
    max_size=1000,
    permutation_num=0,  # ssGSEA no necesita permutaciones para los scores
    threads=4,
    outdir=str(SS_DIR),
    seed=6,
    verbose=True,
)

# Matriz rutas x muestras
scores = ss_res.res2d.pivot_table(index="Term", columns="Name", values="NES")
scores = scores[CONTROL + KO]  # ordenar columnas
scores.to_csv(SS_DIR / "ssgsea_scores.csv")
print(f"Matriz ssGSEA: {scores.shape[0]} rutas x {scores.shape[1]} muestras")
scores.head()

In [ ]:
# Heatmap de rutas x muestras (ordenado por diferencia media KO - Control)
diff = scores[KO].mean(axis=1) - scores[CONTROL].mean(axis=1)
scores_ord = scores.loc[diff.sort_values().index]

fig, ax = plt.subplots(figsize=(8, max(6, 0.28 * len(scores_ord))))
sns.heatmap(scores_ord, cmap="RdBu_r", center=0, ax=ax,
            yticklabels=True, cbar_kws={"label": "ssGSEA NES"})
ax.axvline(len(CONTROL), color="k", lw=2)
ax.set_title("ssGSEA Hallmark — actividad de rutas por muestra\n(Control | KO)")
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(SS_DIR / "heatmap_ssgsea.png", dpi=150)
plt.show()

In [ ]:
# Test estadístico ruta a ruta: Control vs KO
from scipy import stats

rows = []
for term in scores.index:
    t, p = stats.ttest_ind(scores.loc[term, KO], scores.loc[term, CONTROL])
    rows.append({"Term": term,
                 "mean_Control": scores.loc[term, CONTROL].mean(),
                 "mean_KO": scores.loc[term, KO].mean(),
                 "diff_KO_Control": scores.loc[term, KO].mean() - scores.loc[term, CONTROL].mean(),
                 "pvalue": p})
ttest_df = pd.DataFrame(rows)
ttest_df["FDR"] = stats.false_discovery_control(ttest_df["pvalue"])
ttest_df = ttest_df.sort_values("pvalue")
ttest_df.to_csv(SS_DIR / "ssgsea_ttest.csv", index=False)

print("Rutas más diferenciales entre KO y Control (ssGSEA, t-test):")
display(ttest_df.head(15))

## 7. Exportación de resultados

Empaquetamos todos los resultados en un ZIP para descargar.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("resultados_analisis_funcional_MYCN", "zip", "resultados")
print(f"ZIP creado: {zip_path}")
files.download(zip_path)

## 8. Preguntas de reflexión

1. **ORA**: ¿qué procesos biológicos dominan entre los genes UP y DOWN en el KO de MYCN? ¿Encajan con la función conocida de MYCN?
2. **Pre-ranked vs clásico**: ¿coinciden los NES y las rutas significativas? ¿Por qué pueden diferir los p-valores si los NES son casi idénticos?
3. **Métrica de ranking**: repite el pre-ranked usando la columna `stat` de DESeq2 como ranking. ¿Cambian las rutas significativas?
4. **Leading edge**: en el heatmap de la ruta MYC/E2F, ¿la mayoría de genes leading-edge van en la misma dirección? ¿Qué significa eso?
5. **ssGSEA**: ¿hay réplicas que se comporten como "outliers" en el heatmap de rutas? ¿Coincide con lo que viste en el PCA inicial?
6. **Limitación**: *MYCN* no está en la matriz. ¿Cómo comprobarías entonces, con estos datos, que el knockout funcionó?

---

**Entrega**: script(s) `.py` o este notebook ejecutado + un PDF con los resultados y las respuestas a las preguntas.